## Droplet adjacency analysis, background correction, and data preparation

This script continues the analysis performed in droplet_segmentation_and_quantification.ipynb.

Using the generated masks and fluorescence quantification data, it determines whether droplets are adjacent and records the results in the quantification tables. Background fluorescence measured outside the droplets and fluorescence values from negative-control experiments are also subtracted in this script. The processed datasets are then compiled and saved as pickle files.

### Import libraries and define functions


In [1]:
# Import required libraries
import os
import re
import warnings
import pickle
import numpy as np
import pandas as pd
from scipy import ndimage as ndi
from cellpose import utils

# Test data
main_dir = os.path.join(os.path.curdir, "Test data")

# Full dataset
# main_dir = os.path.join(os.path.curdir, "260426 Data") 

analyzed_dir_name = 'Analyzed'

def load_and_prepare_2(main_dir, collection_folder, date, analyzed_dir_name, encoding='shift_jis', subtract=True):
    """Load quantified droplet data, optionally subtract background fluorescence, and add metadata columns."""
    base = os.path.join(main_dir, collection_folder, date, analyzed_dir_name)
    d = pd.read_csv(os.path.join(base, 'data_all.csv'), encoding=encoding)
    if subtract:
        try:
            d_out = pd.read_csv(os.path.join(base, 'outside.csv'), encoding=encoding)
            d = subtract_outside(d, d_out)
        except FileNotFoundError:
            warnings.warn("outside.csv not found; skipping background subtraction")
    d['Sample_Number'] = d['Filename'].str.extract(r'(#\d+)')
    d.loc[:, 'Date'] = d['Filename'].str[-6:]
    return d

def subtract_outside(d, d_outside):
    """Subtract background fluorescence measured outside droplets for each image."""
    for filename in d['Filename'].unique():
        outside = d_outside[d_outside['Filename'] == filename]['Outside'].values[0]
        mask = d['Filename'] == filename
        d.loc[mask, 'Fl_Int-Out'] = d.loc[mask, 'Fl_Int'] - outside
    return d

def filter_droplets(data, sample_num, droplet_type):
    """Filter droplets by sample number and droplet type."""
    if droplet_type in ['Rep', 'N_Rep']:
        filtered = data[
            (data['Sample_Number'] == sample_num)
            & data['Droplet'].str.contains('Green')
            & ~data['Droplet'].str.contains('No')
        ].copy()
    else:  # NDK, N_NDK
        filtered = data[
            (data['Sample_Number'] == sample_num)
            & data['Droplet'].str.contains('NoGreen')
        ].copy()
    filtered.loc[:, 'Droplet'] = droplet_type
    return filtered

def load_masks(directory):
    """Load all segmentation masks from a directory."""
    masks = {}
    mask_files = [f for f in os.listdir(directory) if f.endswith('.npy')]
    for filename in mask_files:
        key = os.path.splitext(filename)[0]
        masks[key] = np.load(os.path.join(directory, filename))
    return masks

def apply_outline_adjacent_with_dilation(data, masks_dict, mask_for_check_key, mask_for_comparison_key):
    """Identify adjacent droplets by first narrowing candidates with dilation and then confirming adjacency using object outlines."""
    data = data.copy()
    data.loc[:, ['adjacent_Candidate', 'adjacent_outline']] = 'x'
    data = adjacent_candidate_labeling(data, masks_dict, mask_for_check_key, mask_for_comparison_key)

    data_close_all = data[data['adjacent_Candidate'].astype(str).str.startswith('o,')].copy()
    data_close_all.loc[:, 'adjacent_Candidate'] = data_close_all['adjacent_Candidate'].str.replace('o, ', '', regex=False).str.replace(r'[\[\]]', '', regex=True)

    candidate_labels_dict_by_row = {}
    for _, row in data_close_all.iterrows():
        candidate_labels = [int(x) for x in str(row['adjacent_Candidate']).split()]
        # Store candidate labels using (Filename, label) tuples as keys.
        candidate_labels_dict_by_row[(row['Filename'], row['label'])] = candidate_labels

    if len(candidate_labels_dict_by_row) == 0:
        return data

    data_close = apply_outline_adjacent(data_close_all, masks_dict, candidate_labels_dict_by_row, mask_for_check_key, mask_for_comparison_key)

    for _, row in data_close.iterrows():
        mask = (data['label'] == row['label']) & (data['Filename'] == row['Filename'])
        data.loc[mask, 'adjacent_outline'] = row['adjacent_outline']
    return data

def adjacent_candidate_labeling(data, masks_dict, mask_for_check_key, mask_for_comparison_key):
    """Identify candidate adjacent droplets using mask dilation and add the results to the data table."""
    # Process each image separately.
    for key in masks_dict:
        if mask_for_check_key not in key:
            continue
        masks_for_check = masks_dict[key]
        masks_for_comparison = masks_dict[key.replace(mask_for_check_key, mask_for_comparison_key)]
        adj_dict = check_mask_adjacent_dilation(masks_for_check, masks_for_comparison)
        adj_df = pd.DataFrame(adj_dict)
        if adj_df.empty:
            continue
        
        # Select rows corresponding to the current image.
        filename = key.replace(f'_masks{mask_for_check_key}', '')
        target_file_mask = (data['Filename'] == filename)
        for _, row in adj_df.iterrows():
            target_label = row['label']
            close_to_list = row['close_to']
            target_rows = target_file_mask & (data['label'] == target_label)
            data.loc[target_rows, 'adjacent_Candidate'] = f'o, {close_to_list}'
    return data

def check_mask_adjacent_dilation(masks_for_check, masks_for_comparison):
    """
    Identify candidate adjacent droplets by dilating masks.
    
    Return: 
    overlap_list, e.g., {'label': [1, 2, ...], 'close_to': [[1 12], [2 5 8], ...]} 
    """
    overlap_list = {'label': [], 'close_to': []}
    labels = np.unique(masks_for_check)
    labels = labels[labels != 0]
    for label in labels:
        mask = (masks_for_check == label)
        dilated_mask = ndi.binary_dilation(mask, iterations=3)
        masks_overlap = masks_for_comparison[dilated_mask]
        if np.any(masks_overlap > 0):
            overlap_list['label'].append(label)
            overlap_list['close_to'].append(np.unique(masks_overlap[masks_overlap != 0]))
    return overlap_list

def apply_outline_adjacent(data, masks_dict, candidate_labels_dict_by_row, mask_for_check_key, mask_for_comparison_key):
    """Confirm droplet adjacency using object outlines and add the results to the data table."""
    data = data.copy()
    adjacent_result = []
    for _, row in data.iterrows():
        filename = row['Filename']
        label = row['label']
        key = (filename, label)
        candidate_labels = candidate_labels_dict_by_row.get(key, [])
        if len(candidate_labels) == 0:
            adjacent_result.append('x')
            continue
        masks_for_check = masks_dict[f"{filename}_masks{mask_for_check_key}"]
        masks_for_comparison = masks_dict[f"{filename}_masks{mask_for_comparison_key}"]
        candidate_labels_dict = {label: [int(cand) for cand in candidate_labels]}
        
        # Record comparison-droplet labels confirmed as adjacent.
        adjacent_labels = check_mask_adjacent_outline_limited(masks_for_check, masks_for_comparison, candidate_labels_dict)
        if len(adjacent_labels) > 0:
            adjacent_result.append(f'o, {adjacent_labels}')
        else:
            adjacent_result.append('x')
    data.loc[:, 'adjacent_outline'] = adjacent_result
    return data

def check_mask_adjacent_outline_limited(masks_for_check, masks_for_comparison, candidate_labels_dict):
    """
    Determine adjacency between droplet outlines using only the candidate labels specified for each target droplet.
    masks_for_check : Labeled masks for the target droplet population.
    masks_for_comparison : Labeled masks for the comparison droplet population.
    candidate_labels_dict : Candidate comparison labels for each target label, identified by mask dilation, e.g,. {1: [1, 2, ...]}.

    Returns:
    adj_list: Labels of comparison droplets confirmed as adjacent.
    """
    adj_list = []    
    labels_for_check = np.unique(masks_for_check)
    labels_for_check = labels_for_check[labels_for_check != 0]
    labels_for_comparison = np.unique(masks_for_comparison)
    labels_for_comparison = labels_for_comparison[labels_for_comparison != 0]

    # Iterate over each target droplet.
    for label_check in labels_for_check:
        candidate_labels = candidate_labels_dict.get(label_check, [])
        if not candidate_labels:
            continue
        _check_mask = np.where(masks_for_check == label_check, label_check, 0)
        outline_check = utils.outlines_list(_check_mask)[0]

        # Identify candidate labels present in the comparison masks.
        comparison_indices = [np.where(labels_for_comparison == int(lab))[0][0] for lab in candidate_labels if int(lab) in labels_for_comparison]
        
        # Compare the target droplet with each candidate droplet.
        for ci in comparison_indices:
            comp_label = labels_for_comparison[ci]
            _comp_mask = np.where(masks_for_comparison == comp_label, comp_label, 0)
            outline_comp = utils.outlines_list(_comp_mask)[0]
            
            # Compare all outline pixels from the two droplets.
            for pixel_check in outline_check:
                for pixel_comp in outline_comp:
                    distance = np.linalg.norm(np.array(pixel_check) - np.array(pixel_comp))
                    # Record droplets whose outlines are within an 8-neighborhood.
                    if distance < 1.42:
                        adj_list.append(int(comp_label))
                        break
                else:
                    # Continue to the next target-outline pixel when no adjacent comparison pixel was found.
                    continue 
                # Stop checking this candidate after adjacency is confirmed.
                break
    return adj_list

def reverse_map_adjacent_outline(source_df, target_df, source_label_col='label', target_label_col='label'):
    """Map adjacency relationships from the source droplets to the corresponding target droplets."""
    target = target_df.copy()
    target.loc[:, 'adjacent_outline'] = 'x'
    for filename in source_df['Filename'].unique():
        sub = source_df[
            (source_df['Filename'] == filename)
            & (source_df['adjacent_outline'].astype(str).str.startswith('o,'))
        ]
        target_to_source = {}
        for _, row in sub.iterrows():
            nums = [int(x) for x in re.findall(r'\d+', str(row['adjacent_outline']))]
            for target_label in nums:
                target_to_source.setdefault(target_label, []).append(int(row[source_label_col]))
        for target_label, source_labels in target_to_source.items():
            mask = (target['Filename'] == filename) & (target[target_label_col] == target_label)
            target.loc[mask, 'adjacent_outline'] = f"o, {sorted(set(source_labels))}"
    return target

def apply_outline_adjacent_with_reverse_mapping(data, target_df, masks_dict, mask_for_check_key, mask_for_comparison_key):
    """Determine adjacency for one droplet population and map the resulting adjacency relationships to the corresponding partner droplets."""
    data = apply_outline_adjacent_with_dilation(data.copy(), masks_dict, mask_for_check_key, mask_for_comparison_key)
    target_adj = reverse_map_adjacent_outline(data.copy(), target_df.copy())
    return data, target_adj

def save_prepared(prepared_dict, save_path):
    """Save a processed data dictionary as a pickle file."""
    with open(save_path, 'wb') as f:
        pickle.dump(prepared_dict, f)

def load_prepared(load_path):
    """Load a processed data dictionary from a pickle file."""
    with open(load_path, 'rb') as f:
        prepared_dict = pickle.load(f)
    return prepared_dict

### Analysis

#### FITC-DEX exchange

In [4]:
# Group definitions: {('#N', 'Date'): 'DEX_label'}
group_configs = {}
# Dictionary of masks
masks_dex_dict = {}

# 10 and 2000 kDa DEX; replicates 1 and 2
d_241024 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '241024', analyzed_dir_name)
group_configs.update({
    ('#1', '241024'): '10k',
    ('#3', '241024'): '2000k',
})
masks_dir = os.path.join(main_dir,'Microscope images for analysis', '241024', analyzed_dir_name, 'Numpy', 'Masks')
masks_dex_dict.update(load_masks(masks_dir))

# 10 and 2000 kDa DEX; replicate 3
d_241029_full = load_and_prepare_2(main_dir, 'Microscope images for analysis', '241029', analyzed_dir_name)
# Exclude the negative control without fluorescent DEX
d_241029 = d_241029_full[~d_241029_full["Filename"].str.startswith("#4")].copy()
# Exclude images in which red droplets were not properly detected
d_241029 = d_241029[~d_241029["Filename"].str.contains('#1.oir_241029|#1_0002.oir_241029|#1_0006.oir_241029')]
group_configs.update({
    ('#1', '241029'): '10k',
    ('#3', '241029'): '2000k',
})
masks_dir = os.path.join(main_dir,'Microscope images for analysis', '241029', analyzed_dir_name, 'Numpy', 'Masks')
masks_dex_dict.update(load_masks(masks_dir))

# 150 kDa DEX at standard and reduced BSA concentrations
d_260409 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '260409', analyzed_dir_name)
group_configs.update({
    ('#1', '260409'): '150k',
    ('#2', '260409'): '150k_BSA01',
})
masks_dir = os.path.join(main_dir,'Microscope images for analysis', '260409', analyzed_dir_name, 'Numpy', 'Masks')
masks_dex_dict.update(load_masks(masks_dir))

d_260413 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '260413', analyzed_dir_name)
group_configs.update({
    ('#1', '260413'): '150k',
    ('#2', '260413'): '150k_BSA01',
})
masks_dir = os.path.join(main_dir,'Microscope images for analysis', '260413', analyzed_dir_name, 'Numpy', 'Masks')
masks_dex_dict.update(load_masks(masks_dir))

d_260415 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '260415', analyzed_dir_name)
group_configs.update({
    ('#1', '260415'): '150k',
    ('#2', '260415'): '150k_BSA01',
})
masks_dir = os.path.join(main_dir,'Microscope images for analysis', '260415', analyzed_dir_name, 'Numpy', 'Masks')
masks_dex_dict.update(load_masks(masks_dir))

d_260609 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '260609', analyzed_dir_name)
# Exclude an image with misaligned acquisition positions between phases
d_260609 = d_260609[~(d_260609['Filename'] == '#4.oir_260609')]
group_configs.update({
    ('#1', '260609'): '10k',
    ('#2', '260609'): '150k',
    ('#3', '260609'): '2000k',
    ('#4', '260609'): '150k_BSA01'
})
masks_dir = os.path.join(main_dir,'Microscope images for analysis', '260609', analyzed_dir_name, 'Numpy', 'Masks')
masks_dex_dict.update(load_masks(masks_dir))

# Background subtraction
d_back = d_241029_full[d_241029_full["Filename"].str.startswith("#4")].copy()
nega = d_back["Fl_Int-Out"].mean()

d_all = pd.concat([d_241024, d_241029, d_260409, d_260413, d_260415, d_260609], ignore_index=True)
d_all["Fl_Int-Out-Back"] = d_all["Fl_Int-Out"] - nega

# Adjacency analysis
prepared_dex = {
    'data': d_all,
    'masks_dict': masks_dex_dict,
}

group_buckets = {}

for (sample_num, date), dex_label in group_configs.items():
    mask_blue = (
        (d_all['Droplet'] == 'Blue')
        & (d_all['Filename'].str.startswith(sample_num))
        & (d_all['Filename'].str.contains(date))
    )
    mask_red = (
        (d_all['Droplet'] == 'Red')
        & (d_all['Filename'].str.startswith(sample_num))
        & (d_all['Filename'].str.contains(date))
    )

    d_blue = d_all[mask_blue].copy()
    d_red = d_all[mask_red].copy()

    if len(d_blue) == 0 and len(d_red) == 0:
        continue

    d_blue, d_red = apply_outline_adjacent_with_reverse_mapping(
        d_blue, d_red, masks_dex_dict, '_blue', '_red'
    )

    d_blue.loc[:, 'DEX'] = dex_label
    d_red.loc[:, 'DEX'] = dex_label

    key_blue = f'd_{dex_label}_blue'
    key_red = f'd_{dex_label}_red'
    group_buckets.setdefault(key_blue, []).append(d_blue)
    group_buckets.setdefault(key_red, []).append(d_red)

ordered_keys = [
    'd_10k_blue', 'd_10k_red',
    'd_150k_blue', 'd_150k_red',
    'd_150k_BSA01_blue', 'd_150k_BSA01_red',
    'd_2000k_blue', 'd_2000k_red',
]

for key in ordered_keys:
    chunks = group_buckets.get(key, [])
    prepared_dex[key] = pd.concat(chunks, ignore_index=True) if len(chunks) > 0 else d_all.iloc[0:0].copy()

#### FAM-RNA exchange

In [6]:
group_configs = {}
masks_fam_rna_dict = {}

# FAM-RNA 1 ('#56') and FAM-RNA 2 ('ZN1') under reduced-BSA conditions
d_241129 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '241129', analyzed_dir_name)
group_configs.update({
    ('#2', '241129'): '#56_BSA01',
    ('#4', '241129'): 'ZN1_BSA01',
})
masks_dir = os.path.join(main_dir,'Microscope images for analysis', '241129', analyzed_dir_name, 'Numpy', 'Masks')
masks_fam_rna_dict.update(load_masks(masks_dir))

# Negative control without FAM-RNA
d_250207 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '250207', analyzed_dir_name)

# Background subtraction
nega = d_250207["Fl_Int-Out"].mean()

d_all = pd.concat([d_241129], ignore_index=True)
d_all["Fl_Int-Out-Back"] = d_all["Fl_Int-Out"] - nega

# Adjacency analysis
prepared_fam_rna = {
    'data': d_all,
    'masks_dict': masks_fam_rna_dict,
}

group_buckets = {}

for (sample_num, date), fam_rna_label in group_configs.items():
    mask_blue = (
        (d_all['Droplet'] == 'Blue')
        & (d_all['Filename'].str.startswith(sample_num))
        & (d_all['Filename'].str.contains(date))
    )
    mask_red = (
        (d_all['Droplet'] == 'Red')
        & (d_all['Filename'].str.startswith(sample_num))
        & (d_all['Filename'].str.contains(date))
    )

    d_blue = d_all[mask_blue].copy()
    d_red = d_all[mask_red].copy()

    if len(d_blue) == 0 and len(d_red) == 0:
        continue

    d_blue, d_red = apply_outline_adjacent_with_reverse_mapping(
        d_blue, d_red, masks_fam_rna_dict, '_blue', '_red'
    )

    d_blue.loc[:, 'FAM-RNA'] = fam_rna_label
    d_red.loc[:, 'FAM-RNA'] = fam_rna_label

    key_blue = f'd_{fam_rna_label}_blue'
    key_red = f'd_{fam_rna_label}_red'
    group_buckets.setdefault(key_blue, []).append(d_blue)
    group_buckets.setdefault(key_red, []).append(d_red)

ordered_keys = [
    'd_#56_BSA01_blue', 'd_#56_BSA01_red',
    'd_ZN1_BSA01_blue', 'd_ZN1_BSA01_red',
]

for key in ordered_keys:
    chunks = group_buckets.get(key, [])
    prepared_fam_rna[key] = pd.concat(chunks, ignore_index=True) if len(chunks) > 0 else d_all.iloc[0:0].copy()

#### TcRR and TcCRR via inter-droplet molecular diffusion

In [8]:
#'Rep_comm': Figs. 4F/G, S12
#'NTP_comm': Figs. 5D/E, S15A/B
#'NTP_syn_Rep_comm': Figs. 5G/H, S15C/D

# Rep_comm
# #1: w/ replicase translation, #2: w/o replicase translation (relabel if not)
d_Rep_comm_N1 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '241119', analyzed_dir_name)
# Remove obvious segmentation artifacts
d_Rep_comm_N1 = d_Rep_comm_N1[~((d_Rep_comm_N1['Filename'] == '#1.oir_241119') & (d_Rep_comm_N1['Droplet'] == 'NoGreen') & (d_Rep_comm_N1['label'].isin([1, 2, 3, 4])))]

d_Rep_comm_N2 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '241203', analyzed_dir_name)
d_Rep_comm_N3 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '241206', analyzed_dir_name)
d_Rep_comm_N4 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '250506', analyzed_dir_name)
d_Rep_comm_N4 = d_Rep_comm_N4[d_Rep_comm_N4['Sample_Number'].isin(['#3', '#4'])].copy()
d_Rep_comm_N4['Sample_Number'] = d_Rep_comm_N4['Sample_Number'].replace({'#3': '#1', '#4': '#2'})
d_Rep_comm = pd.concat([d_Rep_comm_N1, d_Rep_comm_N2, d_Rep_comm_N3, d_Rep_comm_N4])
d_Rep_comm = d_Rep_comm[d_Rep_comm['Sample_Number'].isin(['#1', '#2'])].copy()

masks_Rep_comm_dict = {}
for date in d_Rep_comm['Date'].unique():
    masks_Rep_comm_dict.update(load_masks(os.path.join(main_dir, 'Microscope images for analysis', date, analyzed_dir_name, 'Numpy', 'Masks')))

# NTP_comm
# #1: w/ NDK translation, #2: w/o NDK translation (relabel if not)
d_NTP_comm_N1 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '251028', analyzed_dir_name)
d_NTP_comm_N2 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '251029', analyzed_dir_name)
d_NTP_comm_N3 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '251030', analyzed_dir_name)
d_NTP_comm = pd.concat([d_NTP_comm_N1, d_NTP_comm_N2, d_NTP_comm_N3])
d_NTP_comm = d_NTP_comm[d_NTP_comm['Sample_Number'].isin(['#1', '#3'])].copy()
d_NTP_comm['Sample_Number'] = d_NTP_comm['Sample_Number'].replace({'#3': '#2'})

masks_NTP_comm_dict = {}
for date in d_NTP_comm['Date'].unique():
    masks_NTP_comm_dict.update(load_masks(os.path.join(main_dir, 'Microscope images for analysis', date, analyzed_dir_name, 'Numpy', 'Masks')))

# NTP_syn_Rep_comm
# #1: w/ CDP addition, #2: w/o CDP addition (relabel if not)
d_NTP_syn_Rep_comm_N1 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '251015', analyzed_dir_name)
d_NTP_syn_Rep_comm_N1 = d_NTP_syn_Rep_comm_N1[d_NTP_syn_Rep_comm_N1['Sample_Number'].isin(['#2', '#4'])].copy()
d_NTP_syn_Rep_comm_N1['Sample_Number'] = d_NTP_syn_Rep_comm_N1['Sample_Number'].replace({'#2': '#1', '#4': '#2'})
d_NTP_syn_Rep_comm_N2 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '251017', analyzed_dir_name)
d_NTP_syn_Rep_comm_N3 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '251024', analyzed_dir_name)
d_NTP_syn_Rep_comm_N4 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '260221', analyzed_dir_name)
d_NTP_syn_Rep_comm_N4 = d_NTP_syn_Rep_comm_N4[d_NTP_syn_Rep_comm_N4['Sample_Number'].isin(['#1', '#9'])].copy()
d_NTP_syn_Rep_comm_N4['Sample_Number'] = d_NTP_syn_Rep_comm_N4['Sample_Number'].replace({'#1': '#1', '#9': '#2'})
d_NTP_syn_Rep_comm_N5 = load_and_prepare_2(main_dir, 'Microscope images for analysis', '260319', analyzed_dir_name)
d_NTP_syn_Rep_comm_N5 = d_NTP_syn_Rep_comm_N5[d_NTP_syn_Rep_comm_N5['Sample_Number'].isin(['#1', '#4'])].copy()
d_NTP_syn_Rep_comm_N5['Sample_Number'] = d_NTP_syn_Rep_comm_N5['Sample_Number'].replace({'#1': '#1', '#4': '#2'})
d_NTP_syn_Rep_comm = pd.concat([d_NTP_syn_Rep_comm_N1, d_NTP_syn_Rep_comm_N2, d_NTP_syn_Rep_comm_N3, d_NTP_syn_Rep_comm_N4, d_NTP_syn_Rep_comm_N5])
d_NTP_syn_Rep_comm = d_NTP_syn_Rep_comm[d_NTP_syn_Rep_comm['Sample_Number'].isin(['#1', '#2'])].copy()

masks_NTP_syn_Rep_comm_dict = {}
for date in d_NTP_syn_Rep_comm['Date'].unique():
    masks_NTP_syn_Rep_comm_dict.update(load_masks(os.path.join(main_dir, 'Microscope images for analysis', date, analyzed_dir_name, 'Numpy', 'Masks')))

# Adjacency analysis
SampleNumber = '#1'
NegaConNumber = '#2'

experiment_map = {
    'Rep_comm': (d_Rep_comm, masks_Rep_comm_dict),
    'NTP_comm': (d_NTP_comm, masks_NTP_comm_dict),
    'NTP_syn_Rep_comm': (d_NTP_syn_Rep_comm, masks_NTP_syn_Rep_comm_dict),
}

prepared_comm = {}

for exp_name, (data_tmp, masks_tmp) in experiment_map.items():
    # 'Rep' and 'NDK' correspond to droplets 1 and 2, respectively. 
    # 'N_Rep' and 'N_NDK' represent the corresponding negative controls.
    d_Rep_tmp = filter_droplets(data_tmp.copy(), SampleNumber, 'Rep')
    d_NDK_tmp = filter_droplets(data_tmp.copy(), SampleNumber, 'NDK')
    dN_Rep_tmp = filter_droplets(data_tmp.copy(), NegaConNumber, 'N_Rep')
    dN_NDK_tmp = filter_droplets(data_tmp.copy(), NegaConNumber, 'N_NDK')

    d_NDK_tmp, d_Rep_tmp = apply_outline_adjacent_with_reverse_mapping(
        d_NDK_tmp, d_Rep_tmp, masks_tmp, '_nogreen', '_green'
    )
    dN_NDK_tmp, dN_Rep_tmp = apply_outline_adjacent_with_reverse_mapping(
        dN_NDK_tmp, dN_Rep_tmp, masks_tmp, '_nogreen', '_green'
    )

    prepared_comm[exp_name] = {
        'data': data_tmp,
        'masks_dict': masks_tmp,
        'd_Rep': d_Rep_tmp,
        'dN_Rep': dN_Rep_tmp,
        'd_NDK': d_NDK_tmp,
        'dN_NDK': dN_NDK_tmp
    }

### Save processed data

In [10]:
# Reduced test dataset
save_dir = os.path.join(os.path.curdir, "Test data", "Quantification data")

# Full dataset
# save_dir = os.path.join(os.path.curdir, "260426 Data", "Quantification data")

save_prepared(prepared_dex, os.path.join(save_dir, 'prepared_data_dex.pkl'))
save_prepared(prepared_fam_rna, os.path.join(save_dir, 'prepared_data_fam_rna.pkl'))
save_prepared(prepared_comm, os.path.join(save_dir, 'prepared_data_comm.pkl'))